In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('train.txt',sep=";",header=None,names=['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [10]:
# # Convert emotion labels into numerical values using a dictionary
emotion_dict = {}

for i, emotion in enumerate(df['emotion'].unique()):
    emotion_dict[emotion] = i

df['emotion'] = df['emotion'].map(emotion_dict)

In [11]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [13]:
# Convert all text data to lowercase
df['text'] = df['text'].str.lower()

In [14]:
df['text']

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15995    i just had a very brief time in the beanbag an...
15996    i am now turning and i feel pathetic that i am...
15997                       i feel strong and good overall
15998    i feel like this was such a rude comment and i...
15999    i know a lot but i feel so stupid because i ca...
Name: text, Length: 16000, dtype: object

In [15]:
# Remove punctuation from text data
import string

df['text'] = df['text'].str.replace(
    f"[{string.punctuation}]", "", regex=True
)

In [16]:
df['text']

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15995    i just had a very brief time in the beanbag an...
15996    i am now turning and i feel pathetic that i am...
15997                       i feel strong and good overall
15998    i feel like this was such a rude comment and i...
15999    i know a lot but i feel so stupid because i ca...
Name: text, Length: 16000, dtype: object

In [17]:
# Remove numbers from text data
df['text'] = df['text'].str.replace(r'\d+', '', regex=True)

In [18]:
# Remove URLs and links from text data
df['text'] = df['text'].str.replace(r'https?://\S+|www\.\S+', '', regex=True)

In [19]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [20]:
# Remove emojis and special characters from text data
df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)

In [21]:
# Import NLTK and the English stop words dataset
import nltk
from nltk.corpus import stopwords

# Download the stop words list (only needed once)
nltk.download('stopwords')

# Store English stop words in a set for faster lookup
stop_words = set(stopwords.words('english'))

# Split each text into words, remove stop words, and join the remaining words
df['text'] = df['text'].apply(
    lambda text: ' '.join(
        word for word in text.split()  # Split text into individual words
        if word not in stop_words      # Keep only words that are not stop words
    )
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Stop words are common words such as the, is, and, and in. For emotion classification

In [22]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [32]:
df.shape

(16000, 2)

In [34]:
# Import train_test_split to divide the dataset
from sklearn.model_selection import train_test_split

# X = input text
# y = target labels (emotions)
# test_size=0.20 means 20% data for testing and 80% for training
# random_state=42 ensures the same split every time
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['emotion'],
    test_size=0.20,
    random_state=42
)

# Check the number of samples in training and testing data
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 12800
Testing samples: 3200


In [35]:
# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Create the CountVectorizer object
cv = CountVectorizer()

# Learn the vocabulary from training text and convert it into word counts
X_train_cv = cv.fit_transform(X_train)

# Convert test text using the same vocabulary learned from training data
X_test_cv = cv.transform(X_test)

# Display the dimensions of the converted data
print("Training data shape:", X_train_cv.shape)
print("Testing data shape:", X_test_cv.shape)

Training data shape: (12800, 13361)
Testing data shape: (3200, 13361)


In [36]:
# Import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer object
tfidf = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF features from training text
X_train_tfidf = tfidf.fit_transform(X_train)

# Apply the learned vocabulary and weights to testing text
X_test_tfidf = tfidf.transform(X_test)

# Display the dimensions of the converted data
print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (12800, 13361)
Testing data shape: (3200, 13361)


In [39]:
# Import the three classification algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

# Create a dictionary containing the models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC()
}

# Store the accuracy of each model for comparison
results = {}

# Train and evaluate each model
for name, model in models.items():

    # Train the model using TF-IDF features and emotion labels
    model.fit(X_train_tfidf, y_train)

    # Predict emotions for the test data
    y_pred = model.predict(X_test_tfidf)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Save accuracy for the final comparison
    results[name] = accuracy

    # Print the model name and its accuracy
    print("=" * 60)
    print("Model:", name)
    print("Accuracy:", round(accuracy, 4))

    # Display precision, recall, F1-score, and support for each emotion
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

Model: Naive Bayes
Accuracy: 0.6609

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200

Model: Logistic Regression
Accuracy: 0.8628

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       946
           1       0.90      0.81      0.86       427
           2       0.90      0.61      0.73       296
           3       0.88      0.47      0.61       113
           4       0.86      0.76      0.81       397
    

In [40]:
# Import Linear Support Vector Machine
from sklearn.svm import LinearSVC

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

# Create the SVM model
svm_model = LinearSVC()

# Train SVM using the TF-IDF training data
svm_model.fit(X_train_tfidf, y_train)

# Predict emotions for the test data
svm_pred = svm_model.predict(X_test_tfidf)

# Calculate and display accuracy
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

# Display precision, recall, and F1-score for each emotion
print(classification_report(y_test, svm_pred, zero_division=0))

SVM Accuracy: 0.891875
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.88      0.88       427
           2       0.83      0.74      0.78       296
           3       0.88      0.70      0.78       113
           4       0.87      0.85      0.86       397
           5       0.89      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200

